# Module 0: Environment Setup

## Learning Objectives
- Create a Medallion Architecture database with DQ schema
- Understand the 4 source systems and their data quality characteristics
- Load raw data exactly as it arrives from each source (no transformation)
- Identify intentional DQ issues seeded for later modules

## Business Scenario

You are the **Data Quality Lead** at a Saudi Arabian holding company. The company operates across multiple subsidiaries and must consolidate customer and transaction data from **4 different source systems** into a single Data Warehouse.

Each source has different schemas, different quality levels, and different business ownership. Your job: build a DQ monitoring framework that catches issues at every layer.

---

> **Role Required:** `ACCOUNTADMIN` (this module only). All subsequent modules use `CORP_DQ_ADMIN`.

> **Time:** ~20 minutes

---
## First Time in Snowflake Notebooks?

If this is your first time using Snowflake Notebooks in Workspaces, here's what you need to know:

### 1. Connect Before Running

Before executing any cell, you must **connect** the notebook to a warehouse:

1. Click the **Connect** button (top-left, next to "Run")
2. Select **COMPUTE_WH** as your warehouse
3. Set **Role** to `ACCOUNTADMIN` (top-right "Choose role" picker)
4. The Connect button will show a green dot when connected

> **No green dot?** Click the dropdown arrow next to Connect. You'll see options like "Shut down kernel", "Restart kernel", and "Select service". If no warehouse is attached, use "Select service" to pick `COMPUTE_WH`.

### 2. Running Cells

- **Run one cell:** Click the ▶ (play) button on the cell, or press `Shift+Enter`
- **Run all cells:** Click the dropdown arrow next to "Run" > "Run all"
- **Cell types:** SQL cells have a `SQL` badge (top-left of cell), Python cells have `Python`

### 3. Cell Names

Each cell has a **name** shown at the top (e.g., `create_database`, `dmf_check_national_id_format`). These are descriptive labels that:

- Help you navigate — you can see at a glance what each cell does
- Enable cross-cell references in Python (e.g., `my_query.to_pandas()` references the SQL cell named `my_query`)
- Are pre-set in all lab notebooks — no need to name them yourself

> **Naming convention:** `md_` = markdown, `sql_` = SQL, `dmf_` = Data Metric Function, `checkpoint_` = verification, `quiz_` = answers.

### 4. SQL vs Python Cells

- **SQL cells** run directly on Snowflake (like running in a worksheet)
- **Python cells** run in a managed Python environment with Snowpark pre-installed
- To change a cell's language: click the language badge (`SQL` / `Python`) at the top-left of the cell

> **Ready?** Set role to `ACCOUNTADMIN`, connect to `COMPUTE_WH`, and run the cells below in order.


---
## Step 1: Create Database and Schemas

In [ ]:
CREATE DATABASE IF NOT EXISTS CORP_DWH
    COMMENT = 'Corporate Data Warehouse - Snowflake DQ Monitoring HOL';

CREATE SCHEMA IF NOT EXISTS CORP_DWH.RAW COMMENT = 'Bronze: raw data as-is from source systems';
CREATE SCHEMA IF NOT EXISTS CORP_DWH.SILVER COMMENT = 'Silver: cleansed, unified, scored (Dynamic Tables)';
CREATE SCHEMA IF NOT EXISTS CORP_DWH.GOLD COMMENT = 'Gold: business-ready dimensions and facts (dbt models)';
CREATE SCHEMA IF NOT EXISTS CORP_DWH.DQ COMMENT = 'Data Quality: DMFs, rules catalog, procedures, alerts';

---
## Step 2: Create Roles

In [ ]:
CREATE ROLE IF NOT EXISTS CORP_DQ_ADMIN;
GRANT USAGE ON DATABASE CORP_DWH TO ROLE CORP_DQ_ADMIN;
GRANT USAGE ON ALL SCHEMAS IN DATABASE CORP_DWH TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.RAW TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.SILVER TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.GOLD TO ROLE CORP_DQ_ADMIN;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_ADMIN;
GRANT CREATE DATA METRIC FUNCTION ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_ADMIN;
GRANT CREATE DYNAMIC TABLE ON SCHEMA CORP_DWH.SILVER TO ROLE CORP_DQ_ADMIN;
GRANT CREATE DYNAMIC TABLE ON SCHEMA CORP_DWH.GOLD TO ROLE CORP_DQ_ADMIN;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE CORP_DQ_ADMIN;
GRANT APPLICATION ROLE SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER TO ROLE CORP_DQ_ADMIN;
GRANT EXECUTE DATA METRIC FUNCTION ON ACCOUNT TO ROLE CORP_DQ_ADMIN;
GRANT DATABASE ROLE SNOWFLAKE.DATA_METRIC_USER TO ROLE CORP_DQ_ADMIN;
GRANT ROLE CORP_DQ_ADMIN TO ROLE ACCOUNTADMIN;

> **What this does:** Creates the CORP_DQ_STEWARD role with read-only access to data + write access to the DQ schema. This role is for data stewards who manage rules but don't administer the pipeline.

In [ ]:
CREATE ROLE IF NOT EXISTS CORP_DQ_STEWARD;
GRANT USAGE ON DATABASE CORP_DWH TO ROLE CORP_DQ_STEWARD;
GRANT USAGE ON ALL SCHEMAS IN DATABASE CORP_DWH TO ROLE CORP_DQ_STEWARD;
GRANT ALL PRIVILEGES ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_STEWARD;
GRANT CREATE DATA METRIC FUNCTION ON SCHEMA CORP_DWH.DQ TO ROLE CORP_DQ_STEWARD;
GRANT SELECT ON ALL TABLES IN SCHEMA CORP_DWH.RAW TO ROLE CORP_DQ_STEWARD;
GRANT SELECT ON ALL TABLES IN SCHEMA CORP_DWH.SILVER TO ROLE CORP_DQ_STEWARD;
GRANT SELECT ON ALL TABLES IN SCHEMA CORP_DWH.GOLD TO ROLE CORP_DQ_STEWARD;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE CORP_DQ_STEWARD;
GRANT EXECUTE DATA METRIC FUNCTION ON ACCOUNT TO ROLE CORP_DQ_STEWARD;
GRANT APPLICATION ROLE SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER TO ROLE CORP_DQ_STEWARD;
GRANT ROLE CORP_DQ_STEWARD TO ROLE CORP_DQ_ADMIN;

---
## Step 3: Enable Cross-Region Inference + Notification Integration

> **ACTION REQUIRED:** Replace `<<YOUR_EMAIL>>` below with your actual Snowflake-verified email address before running this cell. The email must match a verified address in your Snowflake account.

In [ ]:
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

CREATE OR REPLACE NOTIFICATION INTEGRATION CORP_DQ_ALERTS
    TYPE = EMAIL ENABLED = TRUE
    ALLOWED_RECIPIENTS = ('<<YOUR_EMAIL>>');
GRANT USAGE ON INTEGRATION CORP_DQ_ALERTS TO ROLE CORP_DQ_ADMIN;

---
## Step 4: Switch to Lab Role

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
## Source System 1: ERP (SAP S/4HANA)

| Attribute | Detail |
|-----------|--------|
| **System** | SAP S/4HANA -- Financial master data |
| **Feed method** | Nightly batch CSV export via SFTP |
| **Data owner** | Finance Department |
| **Quality level** | HIGH -- structured, validated at input |
| **Refresh frequency** | Daily (overnight batch) |

**Business context:** ERP is the "system of record" for billing. If a customer exists in ERP, they receive invoices. IBAN is mandatory because payments are initiated from this system. This is your most trusted source.

**Known issues (minor):**
- One record has leading/trailing spaces in NATIONAL_ID (copy-paste artifact)
- One record is missing the English name (Arabic-only entry)
- Phone format varies: some have +966, others have local 05 prefix

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_CUSTOMERS_ERP (
    RAW_ID NUMBER AUTOINCREMENT,
    CUSTOMER_NAME_AR STRING COMMENT 'Arabic name from SAP',
    CUSTOMER_NAME_EN STRING COMMENT 'English name (optional in SAP)',
    NATIONAL_ID STRING COMMENT 'Saudi ID - should be 10 digits',
    IBAN STRING COMMENT 'Bank account for payments',
    EMAIL STRING,
    PHONE STRING COMMENT 'Format varies: +966, 05, 00966',
    CITY_CODE STRING COMMENT 'SAP city code (RUH, JED, DMM, MKH)',
    CREATED_IN_SOURCE STRING COMMENT 'Original creation date in SAP',
    LOADED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_CUSTOMERS_ERP
    (CUSTOMER_NAME_AR, CUSTOMER_NAME_EN, NATIONAL_ID, IBAN, EMAIL, PHONE, CITY_CODE, CREATED_IN_SOURCE, SOURCE_FILE)
VALUES
    -- Original 6 records (with seeded DQ issues)
    ('Abdullah AR', 'Abdullah Al-Rashid', '1087654321', 'SA0380000000608010167519', 'a.rashid@acme.sa', '+966501234567', 'RUH', '2024-01-15', 'ERP_BATCH_001.csv'),
    ('Fatima AR', 'Fatima Al-Zahrani', '1098765432', 'SA4420000001234567891234', 'f.zahrani@acme.sa', '+966512345678', 'JED', '2024-02-20', 'ERP_BATCH_001.csv'),
    ('Mohammed AR', 'Mohammed Al-Otaibi', ' 1076543210 ', 'SA6680000000608010167520', 'mo.otaibi@acme.sa', '0523456789', 'DMM', '2024-03-10', 'ERP_BATCH_002.csv'),
    ('Nora AR', NULL, '2087654321', 'SA7780000000608010167521', 'n.shamari@acme.sa', '00966534567890', 'RUH', '15/04/2024', 'ERP_BATCH_002.csv'),
    ('Khalid AR', 'Khalid Al-Harbi', '1065432109', 'SA8880000000608010167522', 'k.harbi@acme.sa', '+966545678901', 'MKH', '2024-05-01', 'ERP_BATCH_003.csv'),
    ('Reem AR', 'Reem Al-Tamimi', '1043210987', 'SA5580000000608010167528', 'r.tamimi@acme.sa', '+966534567891', 'RUH', '2024-01-01', 'ERP_BATCH_001.csv'),
    -- Additional 24 clean records
    ('Ahmed AR', 'Ahmed Al-Subaie', '1012345678', 'SA1180000000608010167530', 'a.subaie@acme.sa', '+966551234567', 'RUH', '2024-01-20', 'ERP_BATCH_001.csv'),
    ('Fahad AR', 'Fahad Al-Dosari', '1023456789', 'SA2280000000608010167531', 'f.dosari@acme.sa', '+966552345678', 'JED', '2024-02-01', 'ERP_BATCH_001.csv'),
    ('Sultan AR', 'Sultan Al-Mutairi', '1034567890', 'SA3380000000608010167532', 's.mutairi@acme.sa', '+966553456789', 'DMM', '2024-02-15', 'ERP_BATCH_001.csv'),
    ('Turki AR', 'Turki Al-Shammari', '1045678901', 'SA4480000000608010167533', 't.shammari@acme.sa', '+966554567890', 'RUH', '2024-03-01', 'ERP_BATCH_002.csv'),
    ('Faisal AR', 'Faisal Al-Qahtani', '1056789012', 'SA5580000000608010167534', 'f.qahtani@acme.sa', '+966555678901', 'JED', '2024-03-15', 'ERP_BATCH_002.csv'),
    ('Bandar AR', 'Bandar Al-Harbi', '1067890123', 'SA6680000000608010167535', 'b.harbi@acme.sa', '+966556789012', 'MKH', '2024-04-01', 'ERP_BATCH_002.csv'),
    ('Nawaf AR', 'Nawaf Al-Anazi', '1078901234', 'SA7780000000608010167536', 'n.anazi@acme.sa', '+966557890123', 'DMM', '2024-04-15', 'ERP_BATCH_002.csv'),
    ('Saud AR', 'Saud Al-Ghamdi', '1089012345', 'SA8880000000608010167537', 's.ghamdi@acme.sa', '+966558901234', 'RUH', '2024-05-01', 'ERP_BATCH_003.csv'),
    ('Hamad AR', 'Hamad Al-Zahrani', '1090123456', 'SA9980000000608010167538', 'h.zahrani@acme.sa', '+966559012345', 'JED', '2024-05-15', 'ERP_BATCH_003.csv'),
    ('Abdulaziz AR', 'Abdulaziz Al-Dossary', '1001234567', 'SA1080000000608010167539', 'az.dossary@acme.sa', '+966550123456', 'DMM', '2024-06-01', 'ERP_BATCH_003.csv'),
    ('Majed AR', 'Majed Al-Otaibi', '1013456789', 'SA2180000000608010167540', 'm.otaibi@acme.sa', '+966561234567', 'RUH', '2024-06-15', 'ERP_BATCH_003.csv'),
    ('Saleh AR', 'Saleh Al-Rashid', '1024567890', 'SA3280000000608010167541', 'sal.rashid@acme.sa', '+966562345678', 'JED', '2024-07-01', 'ERP_BATCH_004.csv'),
    ('Nasser AR', 'Nasser Al-Shamrani', '1035678901', 'SA4380000000608010167542', 'n.shamrani@acme.sa', '+966563456789', 'DMM', '2024-07-15', 'ERP_BATCH_004.csv'),
    ('Meshal AR', 'Meshal Al-Tamimi', '1046789012', 'SA5480000000608010167543', 'me.tamimi@acme.sa', '+966564567890', 'RUH', '2024-08-01', 'ERP_BATCH_004.csv'),
    ('Badr AR', 'Badr Al-Shehri', '1057890123', 'SA6580000000608010167544', 'b.shehri@acme.sa', '+966565678901', 'JED', '2024-08-15', 'ERP_BATCH_004.csv'),
    ('Noura AR', 'Noura Al-Harbi', '2012345678', 'SA7680000000608010167545', 'no.harbi@acme.sa', '+966566789012', 'RUH', '2024-09-01', 'ERP_BATCH_004.csv'),
    ('Lama AR', 'Lama Al-Qahtani', '2023456789', 'SA8780000000608010167546', 'l.qahtani@acme.sa', '+966567890123', 'JED', '2024-09-15', 'ERP_BATCH_005.csv'),
    ('Maha AR', 'Maha Al-Dosari', '2034567890', 'SA9880000000608010167547', 'maha.dosari@acme.sa', '+966568901234', 'DMM', '2024-10-01', 'ERP_BATCH_005.csv'),
    ('Dalal AR', 'Dalal Al-Anazi', '2045678901', 'SA1980000000608010167548', 'd.anazi@acme.sa', '+966569012345', 'RUH', '2024-10-15', 'ERP_BATCH_005.csv'),
    ('Haifa AR', 'Haifa Al-Ghamdi', '2056789012', 'SA2080000000608010167549', 'h.ghamdi@acme.sa', '+966570123456', 'JED', '2024-11-01', 'ERP_BATCH_005.csv'),
    ('Asma AR', 'Asma Al-Shammari', '2067890123', 'SA3180000000608010167550', 'as.shammari@acme.sa', '+966571234567', 'DMM', '2024-11-15', 'ERP_BATCH_005.csv'),
    ('Rawan AR', 'Rawan Al-Mutairi', '2078901234', 'SA4280000000608010167551', 'r.mutairi@acme.sa', '+966572345678', 'RUH', '2024-12-01', 'ERP_BATCH_005.csv'),
    ('Shahad AR', 'Shahad Al-Zahrani', '2089012345', 'SA5380000000608010167552', 'sh.zahrani@acme.sa', '+966573456789', 'JED', '2024-12-15', 'ERP_BATCH_005.csv'),
    ('Ghada AR', 'Ghada Al-Otaibi', '1002345678', 'SA6480000000608010167553', 'g.otaibi@acme.sa', '+966574567890', 'DMM', '2025-01-01', 'ERP_BATCH_005.csv');

---
## Source System 2: CRM (Salesforce)

| Attribute | Detail |
|-----------|--------|
| **System** | Salesforce Sales Cloud |
| **Feed method** | API sync every 2 hours (JSON payloads) |
| **Data owner** | Sales Department |
| **Quality level** | LOW -- sales reps enter data quickly, skip optional fields |
| **Refresh frequency** | Every 2 hours (near-real-time) |

**Business context:** CRM captures leads, contacts, and opportunities. Many records are "soft" -- a salesperson adds a contact from a business card or phone call. National ID is not mandatory in Salesforce (it's not needed for sales activities). No IBAN because CRM doesn't handle payments.

**Known issues (significant):**
- 10 out of 25 records have NULL National IDs (field not mandatory in Salesforce)
- 2 records are duplicates of ERP customers (same person registered separately in CRM with personal email)
- City names are free-text (not standardized codes like ERP)
- Multiple records have lowercase city names ('tabuk', 'makkah', 'jeddah')

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_CUSTOMERS_CRM (
    RAW_ID NUMBER AUTOINCREMENT,
    FULL_NAME STRING COMMENT 'Single name field (no AR/EN split)',
    NATIONAL_ID STRING COMMENT 'Often NULL - not mandatory in SF',
    EMAIL STRING,
    MOBILE STRING COMMENT 'Local format (05xxxxxxxx)',
    CITY STRING COMMENT 'Free text - not standardized',
    REGISTRATION_DATE STRING COMMENT 'String date from API',
    LOADED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_CUSTOMERS_CRM
    (FULL_NAME, NATIONAL_ID, EMAIL, MOBILE, CITY, REGISTRATION_DATE, SOURCE_FILE)
VALUES
    -- Original 6 records (with seeded DQ issues: 3 NULL IDs, 2 duplicates, bad city casing)
    ('Sara Al-Dosari', NULL, 's.dosari@acme.sa', '0556789012', 'Riyadh', '2024-06-15', 'CRM_Q2.json'),
    ('Omar Al-Qahtani', NULL, 'o.qahtani@acme.sa', '0567890123', 'Jeddah', '2024-07-20', 'CRM_Q2.json'),
    ('Huda Al-Shehri', NULL, 'h.shehri@acme.sa', '0578901234', 'tabuk', '2024-08-10', 'CRM_Q3.json'),
    ('Abdullah Al-Rashid', '1087654321', 'abdullah.r@example.com', '0501234567', 'Riyadh', '2024-01-20', 'CRM_Q1.json'),
    ('Mohammed Al-Otaibi', '1076543210', 'mohammed.o@example.com', '0523456789', 'Dammam', '2024-03-15', 'CRM_Q1.json'),
    ('Nouf Al-Qahtani', '2043210987', 'n.qahtani2@acme.sa', '578901235', 'Jeddah', '2025-01-01', 'CRM_Q4.json'),
    -- Additional 19 records (mixed quality: some NULLs, some valid)
    ('Hind Al-Mutairi', NULL, 'h.mutairi@acme.sa', '0581234567', 'Riyadh', '2024-04-10', 'CRM_Q2.json'),
    ('Maram Al-Harbi', NULL, 'maram.h@acme.sa', '0582345678', 'Jeddah', '2024-05-20', 'CRM_Q2.json'),
    ('Wafa Al-Ghamdi', NULL, 'w.ghamdi@acme.sa', '0583456789', 'Dammam', '2024-06-01', 'CRM_Q2.json'),
    ('Abeer Al-Shamrani', '1025678901', 'abeer.sh@acme.sa', '0584567890', 'Riyadh', '2024-07-05', 'CRM_Q3.json'),
    ('Dina Al-Anazi', '1036789012', 'dina.a@acme.sa', '0585678901', 'Khobar', '2024-07-15', 'CRM_Q3.json'),
    ('Turki Al-Dosari', '1047890123', 't.dosari@acme.sa', '0586789012', 'Madinah', '2024-08-01', 'CRM_Q3.json'),
    ('Faisal Al-Harbi', NULL, 'faisal.h@example.com', '0587890123', 'makkah', '2024-08-20', 'CRM_Q3.json'),
    ('Bandar Al-Shehri', '1069012345', 'bandar.s@acme.sa', '0588901234', 'Tabuk', '2024-09-01', 'CRM_Q3.json'),
    ('Nawaf Al-Zahrani', '1070123456', 'nawaf.z@acme.sa', '0589012345', 'Abha', '2024-09-15', 'CRM_Q3.json'),
    ('Saud Al-Otaibi', NULL, 's.otaibi@example.com', '0590123456', 'Najran', '2024-10-01', 'CRM_Q4.json'),
    ('Hamad Al-Tamimi', '1092345678', 'hamad.t@acme.sa', '0591234567', 'Jubail', '2024-10-15', 'CRM_Q4.json'),
    ('Majed Al-Shamrani', '1003456789', 'majed.sh@acme.sa', '0592345678', 'Riyadh', '2024-11-01', 'CRM_Q4.json'),
    ('Saleh Al-Mutairi', '1015678901', 'saleh.m@acme.sa', '0593456789', 'Jeddah', '2024-11-15', 'CRM_Q4.json'),
    ('Nasser Al-Qahtani', NULL, 'nasser.q@acme.sa', '0594567890', 'Dammam', '2024-12-01', 'CRM_Q4.json'),
    ('Meshal Al-Ghamdi', '1037890123', 'meshal.g@acme.sa', '0595678901', 'Riyadh', '2024-12-15', 'CRM_Q4.json'),
    ('Badr Al-Dosari', '1048901234', 'badr.d@acme.sa', '0596789012', 'Khobar', '2025-01-05', 'CRM_Q4.json'),
    ('Rawan Al-Shammari', '2058901234', 'rawan.sh@acme.sa', '0597890123', 'Madinah', '2025-01-10', 'CRM_Q4.json'),
    ('Shahad Al-Harbi', '2069012345', 'shahad.h@acme.sa', '0598901234', 'Riyadh', '2025-01-15', 'CRM_Q4.json'),
    ('Ghada Al-Anazi', NULL, 'ghada.a@acme.sa', '0599012345', 'jeddah', '2025-01-20', 'CRM_Q4.json');

---
## Source System 3: Government Portal (GOSI / Absher)

| Attribute | Detail |
|-----------|--------|
| **System** | GOSI (Social Insurance) and Absher (Residency) portal exports |
| **Feed method** | Manual monthly Excel upload by Compliance team |
| **Data owner** | Compliance / Legal Department |
| **Quality level** | VERY LOW -- PDF-to-Excel conversion introduces OCR artifacts |
| **Refresh frequency** | Monthly (manual process) |

**Business context:** Saudi holding companies must cross-reference their records against government registries. GOSI provides social insurance data for Saudi employees; Absher provides residency data for expat workers. The compliance team downloads PDFs from government portals, converts them to Excel (often via OCR), and uploads the result. This process introduces severe formatting errors.

**Known issues (critical -- all records have invalid National IDs):**
- `98765` -- too short (5 digits instead of 10, OCR truncated leading digits)
- `30876543210` -- too long (11 digits, OCR merged two fields)
- `ABC1234567` -- contains letters (OCR misread Arabic digits as Latin characters)

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_GOV_PORTAL (
    RAW_ID NUMBER AUTOINCREMENT,
    PERSON_NAME STRING COMMENT 'Name from government record',
    NATIONAL_ID STRING COMMENT 'Severely corrupted by OCR/PDF extraction',
    EMAIL STRING,
    PHONE STRING,
    GOV_SERVICE STRING COMMENT 'Which government service (GOSI, ABSHER)',
    EXTRACT_DATE STRING COMMENT 'Date of the government extract',
    LOADED_AT TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_GOV_PORTAL
    (PERSON_NAME, NATIONAL_ID, EMAIL, PHONE, GOV_SERVICE, EXTRACT_DATE, SOURCE_FILE)
VALUES
    -- Original 3 records (all have INVALID National IDs from OCR corruption)
    ('Tariq Al-Mutairi', '98765', 't.mutairi@acme.sa', '+966589012345', 'GOSI', '2024-09-01', 'GOV_EXTRACT_SEP.xlsx'),
    ('Layla Al-Ghamdi', '30876543210', 'l.ghamdi@acme.sa', '+966590123456', 'ABSHER', '2024-09-15', 'GOV_EXTRACT_SEP.xlsx'),
    ('Yousef Al-Dossary', 'ABC1234567', 'y.dossary@acme.sa', '+966501234568', 'GOSI', '2024-10-01', 'GOV_EXTRACT_OCT.xlsx'),
    -- Additional 7 records (valid government data)
    ('Ahmed Al-Subaie', '1012345678', 'a.subaie@acme.sa', '+966551234567', 'GOSI', '2024-10-15', 'GOV_EXTRACT_OCT.xlsx'),
    ('Fahad Al-Dosari', '1023456789', 'f.dosari@acme.sa', '+966552345678', 'ABSHER', '2024-11-01', 'GOV_EXTRACT_NOV.xlsx'),
    ('Sultan Al-Mutairi', '1034567890', 's.mutairi@acme.sa', '+966553456789', 'GOSI', '2024-11-15', 'GOV_EXTRACT_NOV.xlsx'),
    ('Faisal Al-Qahtani', '1056789012', 'f.qahtani@acme.sa', '+966555678901', 'ABSHER', '2024-12-01', 'GOV_EXTRACT_DEC.xlsx'),
    ('Nawaf Al-Anazi', '1078901234', 'n.anazi@acme.sa', '+966557890123', 'GOSI', '2024-12-15', 'GOV_EXTRACT_DEC.xlsx'),
    ('Noura Al-Harbi', '2012345678', 'no.harbi@acme.sa', '+966566789012', 'ABSHER', '2025-01-01', 'GOV_EXTRACT_JAN.xlsx'),
    ('Lama Al-Qahtani', '2023456789', 'l.qahtani@acme.sa', '+966567890123', 'GOSI', '2025-01-15', 'GOV_EXTRACT_JAN.xlsx');

---
## Source System 4: Bank Transaction Feed

| Attribute | Detail |
|-----------|--------|
| **System** | Core banking reconciliation file |
| **Feed method** | Daily SFTP drop (CSV) by Treasury |
| **Data owner** | Treasury Department |
| **Quality level** | MEDIUM -- structured but dates/amounts arrive as strings |
| **Refresh frequency** | Daily (morning drop, must be processed by noon) |

**Business context:** Bank feeds are the financial truth. Every SAR must reconcile with the general ledger. But the raw format from the bank requires parsing: dates come in multiple formats, amounts include commas and currency prefixes, and the bank uses single-character type codes.

**Known issues:**
- Row 5: BLANK customer reference (empty string, not NULL -- subtle!)
- Row 6: Negative amount (incorrectly coded refund)
- Row 4: Stale record loaded 3 days ago (SLA is 2 hours)
- Mixed date formats: ISO (2025-08-15) and DD/MM/YYYY (15/08/2025)
- Amounts as strings: '7,500.50' and 'SAR 22000'

In [ ]:
CREATE OR REPLACE TABLE CORP_DWH.RAW.STG_TRANSACTIONS (
    RAW_ID NUMBER AUTOINCREMENT,
    CUSTOMER_REF STRING COMMENT 'Customer reference from bank',
    TXN_DATE_STR STRING COMMENT 'Date as STRING - mixed formats!',
    AMOUNT_STR STRING COMMENT 'Amount as STRING - has commas, prefix!',
    CURRENCY STRING,
    TXN_TYPE_CODE STRING COMMENT 'Single char: P=Payment, I=Invoice, T=Transfer, R=Refund',
    LOADED_AT TIMESTAMP_LTZ COMMENT 'When we received the file',
    SOURCE_FILE STRING
);

INSERT INTO CORP_DWH.RAW.STG_TRANSACTIONS
    (CUSTOMER_REF, TXN_DATE_STR, AMOUNT_STR, CURRENCY, TXN_TYPE_CODE, LOADED_AT, SOURCE_FILE)
VALUES
    -- Original 6 records (with seeded DQ issues: blank ref, negative amount, stale record)
    ('CUST-001', '2025-08-15', '15000.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_001.csv'),
    ('CUST-002', '15/08/2025', '7,500.50', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_001.csv'),
    ('CUST-003', '2025-08-14', 'SAR 22000', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_001.csv'),
    ('CUST-004', '2025-08-13', '3200.00', 'SAR', 'R', DATEADD(DAY, -3, CURRENT_TIMESTAMP()), 'BANK_FEED_001.csv'),
    ('', '2025-08-16', '8900.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_001.csv'),
    ('CUST-001', '2025-08-16', '-500.00', 'SAR', 'R', CURRENT_TIMESTAMP(), 'BANK_FEED_001.csv'),
    -- Additional 44 records (realistic amounts, varied dates, all 4 txn types)
    ('CUST-005', '2025-08-01', '2,750.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-006', '2025-08-01', '4100.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-007', '01/08/2025', '6,200.75', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-008', '2025-08-02', '1500.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-009', '2025-08-02', 'SAR 9800', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-010', '2025-08-03', '12,450.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-001', '2025-08-03', '3300.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-002', '03/08/2025', '5,600.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-011', '2025-08-04', '7800.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-012', '2025-08-04', '2,100.50', 'SAR', 'R', CURRENT_TIMESTAMP(), 'BANK_FEED_002.csv'),
    ('CUST-013', '2025-08-05', 'SAR 18500', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-014', '05/08/2025', '4,750.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-015', '2025-08-05', '950.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-003', '2025-08-06', '6,300.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-016', '2025-08-06', '11,200.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-017', '2025-08-07', '3,400.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-018', '07/08/2025', '8,900.25', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-019', '2025-08-07', '1,200.00', 'SAR', 'R', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-020', '2025-08-08', 'SAR 5500', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-005', '2025-08-08', '14,300.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_003.csv'),
    ('CUST-021', '2025-08-09', '2,850.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-022', '09/08/2025', '7,600.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-023', '2025-08-09', '4,500.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-024', '2025-08-10', '9,100.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-025', '2025-08-10', '1,800.00', 'SAR', 'R', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-006', '2025-08-11', '6,750.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-007', '11/08/2025', '3,200.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-008', '2025-08-11', 'SAR 45000', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-026', '2025-08-12', '5,400.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-027', '2025-08-12', '2,300.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_004.csv'),
    ('CUST-028', '2025-08-13', '8,100.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-029', '13/08/2025', '10,500.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-030', '2025-08-13', '3,750.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-001', '2025-08-14', '7,200.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-010', '2025-08-14', '4,800.00', 'SAR', 'R', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-015', '2025-08-15', '6,100.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-020', '15/08/2025', '11,750.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-025', '2025-08-15', 'SAR 3900', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-002', '2025-08-16', '5,000.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-012', '2025-08-16', '2,600.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    -- Outliers for anomaly detection (Module 5)
    ('CUST-003', '2025-08-17', '87,500.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-009', '2025-08-17', '92,000.00', 'SAR', 'P', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-014', '17/08/2025', '89,300.00', 'SAR', 'T', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv'),
    ('CUST-018', '2025-08-18', '4,200.00', 'SAR', 'I', CURRENT_TIMESTAMP(), 'BANK_FEED_005.csv');

---
## Verification: Row Counts

In [ ]:
SELECT 'RAW.STG_CUSTOMERS_ERP' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM CORP_DWH.RAW.STG_CUSTOMERS_ERP
UNION ALL SELECT 'RAW.STG_CUSTOMERS_CRM', COUNT(*) FROM CORP_DWH.RAW.STG_CUSTOMERS_CRM
UNION ALL SELECT 'RAW.STG_GOV_PORTAL', COUNT(*) FROM CORP_DWH.RAW.STG_GOV_PORTAL
UNION ALL SELECT 'RAW.STG_TRANSACTIONS', COUNT(*) FROM CORP_DWH.RAW.STG_TRANSACTIONS
ORDER BY TABLE_NAME;

---
## Setup Complete

You now have **RAW data only** -- exactly as it arrives from each source system. No transformations have been applied yet.

| Source | Table | Rows | Quality | Owner |
|--------|-------|------|---------|-------|
| SAP ERP | STG_CUSTOMERS_ERP | 30 | High | Finance |
| Salesforce CRM | STG_CUSTOMERS_CRM | 25 | Low | Sales |
| Gov Portal (GOSI/Absher) | STG_GOV_PORTAL | 10 | Very Low | Compliance |
| Bank Feed | STG_TRANSACTIONS | 50 | Medium | Treasury |

**Next:** Open `0B_DATA_PIPELINE` to build the transformation pipeline using Dynamic Tables and dbt.